# CDEA | Main Study + Small-Model Extension

**Baseline vs Proposed (CDEA)** in 32 fixed OpenApps tasks.

Large (8B-9B) outcomes come **only from the Main Study**. Small (3B-4B) outcomes come **only from the Small subset of the Scale Study**. The 128 separately executed Scale Large pairs remain archived, but are **excluded** from every derived analysis and trace summary. No source record or reward is rewritten.

Because Small and Large come from different controller revisions, size differences are exploratory, not isolated causal effects of parameter count. Keep the two input ZIPs next to the notebook (or upload to Colab Files), then Run all. Every exported PNG is a separate chart.


## 01. Input selection, coverage and statistics

Original inputs: Main (128 task-model pairs; four Large checkpoints) and Scale (256 pairs; eight checkpoints). For this **derived analysis view**, use **128 Main Large + 128 Scale Small = 256 selected pairs**. Do not double-count Main by adding it again as a separate replicate, and do not include the raw Scale Large pairs in the denominator.

Invalid and not-started pairs are reported separately. Paired effect sizes, exact McNemar tests and task-resampled sensitivity intervals are exploratory.


In [ ]:
from pathlib import Path
from collections import Counter
import io, json, zipfile, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binomtest
from IPython.display import display

# Leave blank to detect archives by manifest protocol. Set explicit paths if needed.
MAIN_ZIP = ''    # e.g. '/content/main_observation_v1(5).zip'
SCALE_ZIP = ''   # e.g. '/content/scale_reference_v4.zip'
BOOTSTRAPS = 4000
SEED = 233
ROOT = Path('cdea_analysis_outputs')
FIG = ROOT / 'figures'
TAB = ROOT / 'tables'
FIG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)
TABLES, FIGURES = {}, []

pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 155)
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 180, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})


def _manifest_in_zip(z):
    roots = [p for p in z.namelist() if p.endswith('manifest.json') and p.count('/') <= 1]
    if len(roots) != 1:
        raise ValueError('Expected one manifest.json at the ZIP root or one directory below it')
    path = roots[0]
    return path[:-len('manifest.json')], json.loads(z.read(path))


def find_archive(protocol, explicit=''):
    accepted = {protocol} if isinstance(protocol, str) else set(protocol)
    if explicit:
        path = Path(explicit).expanduser().resolve()
        if not path.is_file(): raise FileNotFoundError(path)
        return path
    # Prefer notebook working directory; only look in fallback locations if missing.
    for folder in (Path.cwd(), Path('/content')):
        if not folder.is_dir(): continue
        matches = []
        for p in sorted(set(p.resolve() for p in folder.glob('*.zip'))):
            try:
                with zipfile.ZipFile(p) as z:
                    _, m = _manifest_in_zip(z)
                    if m.get('protocol') in accepted: matches.append(p)
            except (OSError, ValueError, KeyError, zipfile.BadZipFile):
                continue
        if len(matches) > 1:
            raise ValueError(f'Multiple {protocol} inputs: {matches}; set MAIN_ZIP or SCALE_ZIP explicitly')
        if matches:return matches[0]
    return None


def as_bool(s):
    out = s.astype('string').str.strip().str.lower().map(
        {'true': True, 'false': False, '1': True, '0': False})
    return out.astype('boolean')


def study_from_zip(path, protocol):
    if path is None: return None
    accepted = {protocol} if isinstance(protocol, str) else set(protocol)
    with zipfile.ZipFile(path) as z:
        prefix, manifest = _manifest_in_zip(z)
        if manifest.get('protocol') not in accepted:
            raise ValueError(f'Unexpected protocol: {manifest.get("protocol")}; expected one of {sorted(accepted)}')
        pairs = pd.read_csv(io.BytesIO(z.read(prefix + 'pairs.csv')), encoding='utf-8-sig')
        episodes = pd.read_csv(io.BytesIO(z.read(prefix + 'results.csv')), encoding='utf-8-sig')
    pairs.columns = pairs.columns.str.lstrip('\ufeff')
    episodes.columns = episodes.columns.str.lstrip('\ufeff')
    for field in ('paired_valid', 'baseline_success', 'proposed_success'):
        pairs[field] = as_bool(pairs[field])
    for field in ('official_success_raw', 'paired_analysis_success', 'episode_valid', 'analysis_included'):
        if field in episodes: episodes[field] = as_bool(episodes[field])
    scheduled = int(manifest['planned_pairs'])
    if len(pairs) != scheduled or len(manifest.get('pairs', [])) != scheduled:
        raise ValueError('Scheduled pair count conflicts with manifest or pairs.csv')
    if len(episodes) > int(manifest['planned_episodes']):
        raise ValueError('More episodes than the declared schedule')
    if pairs.pair_id.duplicated().any() or episodes.duplicated(['pair_id','method']).any():
        raise ValueError('Duplicate pair_id or episode (pair_id,method) slot')
    if set(pairs.pair_id) != {p['pair_id'] for p in manifest['pairs']}:
        raise ValueError('Pair roster does not match the manifest')
    if not set(episodes.method.dropna()).issubset({'Baseline','Proposed'}):
        raise ValueError('Unexpected treatment labels in results.csv')
    if pairs.paired_valid.isna().any():
        raise ValueError('Missing paired_valid status')
    valid = pairs.loc[pairs.paired_valid.eq(True)].copy()
    if valid[['baseline_success','proposed_success']].isna().any().any():
        raise ValueError('Valid pair has missing success outcome')
    valid['task_id'] = valid.task_group.astype(str) + '/' + valid.task.astype(str)
    valid['paired_delta'] = valid.proposed_success.astype(int) - valid.baseline_success.astype(int)
    # Check that the pair outcome and the two recorded episode outcomes agree.
    included = episodes[episodes.pair_id.isin(valid.pair_id)]
    if len(included) != 2 * len(valid):
        raise ValueError('Each valid pair requires two completed episode slots')
    pv = included.pivot(index='pair_id', columns='method', values='paired_analysis_success')
    indexed = valid.set_index('pair_id')
    if not indexed.index.isin(pv.index).all() or not {'Baseline','Proposed'}.issubset(pv.columns):
        raise ValueError('Missing Baseline / Proposed episode for a valid pair')
    if not (pv.loc[indexed.index, 'Baseline'].eq(indexed.baseline_success).all() and
            pv.loc[indexed.index, 'Proposed'].eq(indexed.proposed_success).all()):
        raise ValueError('Pair outcomes disagree with stored episode success')
    if {'official_success_raw','paired_analysis_success'}.issubset(included.columns):
        if not included.official_success_raw.eq(included.paired_analysis_success).all():
            raise ValueError('Official and paired endpoint flags disagree')
    if 'analysis_included' in included and not included.analysis_included.eq(True).all():
        raise ValueError('Valid episodes have analysis_included != true')
    not_started = pairs.pair_status.astype(str).str.contains('not_started|unstarted', case=False, regex=True)
    counts = {'planned_pairs': scheduled, 'valid_pairs': len(valid),
              'invalid_pairs': int((~pairs.paired_valid.fillna(False) & ~not_started).sum()),
              'not_started_pairs': int(not_started.sum()),
              'episode_records': len(episodes),
              'planned_episodes': int(manifest['planned_episodes'])}
    return {'path': path, 'manifest': manifest, 'pairs': pairs, 'valid': valid,
            'episodes': episodes, 'prefix': prefix, 'counts': counts,
            'sha256': hashlib.sha256(path.read_bytes()).hexdigest()}


def ci_task_bootstrap(df, value='paired_delta', draws=BOOTSTRAPS):
    if df.empty: return (np.nan, np.nan)
    g = df.groupby('task_id')[value].agg(['sum','count']).to_numpy(float)
    if len(g) < 2: return (np.nan, np.nan)
    idx = np.random.default_rng(SEED).integers(0, len(g), (draws, len(g)))
    pick = g[idx]
    vals = 100 * pick[:,:,0].sum(axis=1) / pick[:,:,1].sum(axis=1)
    return tuple(np.percentile(vals,[2.5,97.5]))


def paired_stats(df):
    n = len(df)
    if n == 0: return None
    b, p = df.baseline_success.eq(True), df.proposed_success.eq(True)
    both, p_only, b_only, neither = int((b & p).sum()), int((~b & p).sum()), int((b & ~p).sum()), int((~b & ~p).sum())
    ci = ci_task_bootstrap(df)
    base, prop = int(b.sum()), int(p.sum())
    assert both+p_only+b_only+neither == n
    return {'pairs': n, 'baseline_n': base, 'cdea_n': prop,
            'baseline_pct':100*base/n, 'cdea_pct':100*prop/n, 'delta_pp':100*(prop-base)/n,
            'relative_gain_pct': 100*(prop/base-1) if base else np.nan,
            'both_success':both, 'cdea_only':p_only, 'baseline_only':b_only, 'neither_success':neither,
            'mcnemar_p':binomtest(min(p_only,b_only),p_only+b_only,0.5).pvalue if p_only+b_only else 1.,
            'task_ci_low':ci[0], 'task_ci_high':ci[1]}


def summarize(study_name, grouping, frame):
    v = paired_stats(frame)
    return {'study':study_name, 'group':grouping, **v} if v is not None else None


def pretty(frame, cols=None, digits=2):
    if frame is None or frame.empty:
        display(pd.DataFrame({'Note':['No matching data']}))
        return
    show = frame[cols].copy() if cols else frame.copy()
    if 'mcnemar_p' in show:
        show['mcnemar_p'] = show['mcnemar_p'].map(
            lambda p: '<0.001' if pd.notna(p) and p < .001 else (f'{p:.3f}' if pd.notna(p) else '-'))
    display(show.round(digits).reset_index(drop=True))


def record_table(name, frame):
    if frame is not None and not frame.empty:
        TABLES[name] = frame.copy()


def save_chart(name, fig):
    path = FIG / (name+'.png')
    fig.savefig(path, dpi=180, bbox_inches='tight', facecolor='white')
    FIGURES.append(path)
    plt.show()
    plt.close(fig)


def grouped_bars(rows, x_col, categories, title, ylabel='Success rate (%)', rotation=0, ylim=None):
    fig, ax = plt.subplots(figsize=(max(7.2,len(rows)*1.22), 4.4))
    x=np.arange(len(rows)); width=.36
    for i,(column, label) in enumerate(categories):
        bars=ax.bar(x+(i-.5)*width,rows[column],width,label=label)
        if len(rows) <= 5:
            ax.bar_label(bars, fmt='%.1f', padding=2, fontsize=8)
    ax.set_xticks(x, rows[x_col], rotation=rotation, ha='right' if rotation else 'center')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    if ylim is not None:ax.set_ylim(*ylim)
    else:ax.axhline(0, lw=.9, c='gray')
    ax.legend(frameon=False, loc='best')
    ax.grid(axis='y', alpha=.18); ax.set_axisbelow(True)
    fig.tight_layout()
    return fig


main=study_from_zip(find_archive('cdea_main_v1',MAIN_ZIP),'cdea_main_v1')
small_protocols=('cdea_small_extension_v1','cdea_scale_v1')
raw_scale=study_from_zip(find_archive(small_protocols,SCALE_ZIP),small_protocols)
if main is None and raw_scale is None:
    raise FileNotFoundError('Upload the Main and Scale ZIPs or set MAIN_ZIP/SCALE_ZIP')

small=None
if raw_scale is not None:
    if 'size' not in raw_scale['pairs']:
        raise ValueError('Scale size labels missing')
    small=dict(raw_scale)
    small['pairs']=raw_scale['pairs'].loc[raw_scale['pairs']['size'].eq('small')].copy()
    small_ids=set(small['pairs']['pair_id'])
    small['valid']=raw_scale['valid'].loc[raw_scale['valid']['size'].eq('small')].copy()
    small['episodes']=raw_scale['episodes'].loc[
        raw_scale['episodes']['pair_id'].isin(small_ids)].copy()
    if len(small['pairs']) != 128:
        raise ValueError('Exactly 128 Small checkpoint pairs must be scheduled')
    if raw_scale['manifest']['protocol'] == 'cdea_scale_v1':
        if raw_scale['counts']['planned_pairs'] != 256:
            raise ValueError('Historical Scale v1 must schedule 256 pairs before selecting Small')
    elif raw_scale['counts']['planned_pairs'] != 128:
        raise ValueError('Small-only extension must schedule exactly 128 pairs')
    if not small['pairs']['size'].eq('small').all():
        raise ValueError('A Large checkpoint appeared in the selected Small cohort')
    if not set(small['valid']['pair_id']).issubset(small_ids):
        raise ValueError('Small validity roster mismatch')
    not_started=small['pairs']['pair_status'].astype(str).str.contains(
        'not_started|unstarted',case=False,regex=True)
    small['counts']={'planned_pairs':len(small['pairs']),
        'valid_pairs':len(small['valid']),
        'invalid_pairs':int((~small['pairs']['paired_valid'].fillna(False)&~not_started).sum()),
        'not_started_pairs':int(not_started.sum()),
        'episode_records':len(small['episodes']),'planned_episodes':2*len(small['pairs'])}
SOURCES={'Main Large':main,'Scale Small':small}
coverage=pd.DataFrame([{'selected_source':name,'archive':study['path'].name,
    **study['counts']} for name,study in SOURCES.items() if study is not None])
record_table('01_source_coverage',coverage)
pretty(coverage)
raw_excluded=len(raw_scale['pairs'])-len(small['pairs']) if small is not None else 0
print('Excluded prior Scale Large pairs:',raw_excluded,
      '(0 for a new Small-only extension archive, 128 for historical Scale v1)')

scale=None
source_mapping=pd.DataFrame()
source_config=pd.DataFrame()
if main is not None and small is not None:
    large=main['valid'].copy()
    large['size']='large'
    large['family']=large['model_label'].astype(str).str.rsplit('-',n=1).str[0]
    large['origin_study']='Main'
    large['origin_archive']=main['path'].name
    large['origin_pair_id']=large['pair_id']
    smaller=small['valid'].copy()
    smaller['family']=smaller['model_label'].astype(str).str.rsplit('-',n=1).str[0]
    smaller['origin_study']='Scale Small'
    smaller['origin_archive']=small['path'].name
    smaller['origin_pair_id']=smaller['pair_id']
    comparison=pd.concat([large,smaller],ignore_index=True)
    comparison['unique_pair_id']=comparison['origin_study']+':'+comparison['pair_id']
    if comparison.unique_pair_id.duplicated().any():
        raise ValueError('Source-specific pair IDs collide')
    if comparison.duplicated(['model_label','task_id']).any():
        raise ValueError('Duplicate task-model observation in derived comparison')
    if set(comparison.family)!={'qwen3.5','ministral3','granite4.1','xlam2'}:
        raise ValueError('Unexpected checkpoint families')
    if set(comparison.loc[comparison['size'].eq('large'),'origin_study'])!={'Main'}:
        raise ValueError('Large outcomes must come exclusively from Main')
    if set(comparison.loc[comparison['size'].eq('small'),'origin_study'])!={'Scale Small'}:
        raise ValueError('Small outcomes must come exclusively from Scale Small')
    roster=set(main['pairs'].task_group.astype(str)+'/'+main['pairs'].task.astype(str))
    for (family,size),grp in comparison.groupby(['family','size']):
        source_pairs=(main if size=='large' else small)['pairs']
        ref=source_pairs.loc[source_pairs.model_label.eq(grp.model_label.iloc[0])]
        if len(ref)!=32 or set(ref.task_group.astype(str)+'/'+ref.task.astype(str))!=roster:
            raise ValueError(f'Nonmatching task roster for {family} {size}')
    scale={'valid':comparison}
    source_mapping=pd.DataFrame([{'size':size,'origin':grp.origin_study.iloc[0],
        'archive':grp.origin_archive.iloc[0],'selected_valid_pairs':len(grp),
        'checkpoint_count':grp.model_label.nunique(),'unique_task_ids':grp.task_id.nunique()}
        for size,grp in comparison.groupby('size')]).sort_values('size')
    pretty(source_mapping)
    record_table('02_model_size_source_mapping',source_mapping)
    source_config=pd.DataFrame([{'source':name,
        'controller_commit':study['manifest'].get('controller_git_commit'),
        'openapps_commit':study['manifest'].get('openapps_git_commit'),
        **{k:study['manifest'].get('shared_parameters',{}).get(k,'not specified')
            for k in ('seed','max_steps','temperature','model_timeout','num_predict','num_ctx')}}
        for name,study in SOURCES.items() if study is not None])
    print('Source configuration comparison; controller revisions differ:')
    pretty(source_config)
    record_table('03_source_configuration_comparison',source_config)
else:
    print('Both original ZIPs are required for the model-size comparison.')
print('Bootstrap task resamples:',BOOTSTRAPS,'seed:',SEED)


## 02. Main Experiment - Primary & Secondary Results

**Main primary endpoint:** the 20 matched bank-information task-model pairs. Also inspect all 128 pairs, individual models, the three task cohorts, and the five bank-information task IDs. McNemar is computed from discordant pairs, not from the two independent binomial proportions. Do not treat the repeated banking tasks as independent population samples.


In [ ]:
main_overview = pd.DataFrame()
main_category = pd.DataFrame()
main_model = pd.DataFrame()
main_bank_task = pd.DataFrame()
main_bank_model = pd.DataFrame()
if main is not None:
    df=main['valid']
    primary=df.loc[df.stratum.eq('bank_information')]
    main_overview=pd.DataFrame([summarize('Main','Bank information (primary)',primary),
                                summarize('Main','All tasks',df)])
    main_category=pd.DataFrame([summarize('Main',k,grp)
                                for k,grp in df.groupby('stratum')])
    main_model=pd.DataFrame([summarize('Main',k,grp)
                             for k,grp in df.groupby('model_label')])
    main_bank_task=pd.DataFrame([summarize('Main',k,grp)
                                 for k,grp in primary.groupby('task_id')])
    main_bank_model=pd.DataFrame([summarize('Main',k,grp)
                                  for k,grp in primary.groupby('model_label')])
    print('MAIN: Primary cohort and overall study')
    pretty(main_overview, ['group','pairs','baseline_n','cdea_n','baseline_pct','cdea_pct',
        'delta_pp','relative_gain_pct','cdea_only','baseline_only','mcnemar_p','task_ci_low','task_ci_high'])
    print('Main categories:')
    pretty(main_category, ['group','pairs','baseline_pct','cdea_pct','delta_pp'])
    print('All four Main checkpoints:')
    pretty(main_model, ['group','pairs','baseline_pct','cdea_pct','delta_pp'])
    print('Five bank-information task IDs (each evaluated using four checkpoints):')
    pretty(main_bank_task, ['group','pairs','baseline_n','cdea_n','delta_pp'])
    print('Bank-information outcomes by checkpoint:')
    pretty(main_bank_model, ['group','pairs','baseline_n','cdea_n','delta_pp'])
    print('Primary four-cell paired table:')
    print('Both success / CDEA only / Baseline only / neither =',
          *main_overview.iloc[0][['both_success','cdea_only','baseline_only','neither_success']].astype(int).to_list())
    for name,t in [('02_main_summary',main_overview),('03_main_by_category',main_category),
                   ('04_main_by_model',main_model),('05_main_bank_by_task',main_bank_task),
                   ('06_main_bank_by_model',main_bank_model)]: record_table(name,t)
else:
    print('No Main ZIP: Main reports and figures are skipped.')


In [ ]:
if main is not None:
    row=main_overview.iloc[0]
    bars=pd.DataFrame({'method':['Baseline','CDEA'], 'success':[row.baseline_pct,row.cdea_pct]})
    fig,ax=plt.subplots(figsize=(6.2,4.3))
    plot=ax.bar(bars.method,bars.success,width=.48)
    ax.bar_label(plot, labels=[f'{int(row.baseline_n)}/{int(row.pairs)} ({row.baseline_pct:.1f}%)',
                               f'{int(row.cdea_n)}/{int(row.pairs)} ({row.cdea_pct:.1f}%)'],padding=5,fontsize=10)
    ax.set(ylim=(0,106), ylabel='Task success rate (%)', title='Main: Bank-information primary endpoint')
    ax.text(.5,-.16,f'Paired difference: {row.delta_pp:+.1f} percentage points | {int(row.pairs)} matched pairs',
            transform=ax.transAxes,ha='center',fontsize=9)
    ax.grid(axis='y',alpha=.15);ax.set_axisbelow(True)
    save_chart('01_main_primary_bank_success',fig)

    category_order=['bank_information','cross_app_workflow','primitive_control']
    categories=main_category.set_index('group').reindex(category_order).reset_index()
    categories['display']=categories['group'].str.replace('_',' ',regex=False).str.title()
    fig=grouped_bars(categories,'display',[('baseline_pct','Baseline'),('cdea_pct','CDEA')],
        'Main: Outcomes across three task categories',ylim=(0,106))
    save_chart('02_main_task_categories',fig)


## 03. Model-size comparison: Main Large versus Scale Small

The 8 checkpoints are drawn from two source studies. Large outcomes for four 8B-9B checkpoints are taken only from Main; Small outcomes for four 3B-4B checkpoints are taken only from the Small subset of Scale. The separately executed Scale Large episodes remain in their original archive but are never used in the derived comparison. The 32 tasks and two policy arms are matched within each checkpoint.


In [ ]:
scale_overview=pd.DataFrame()
scale_checkpoints=pd.DataFrame()
scale_family=pd.DataFrame()
if scale is not None:
    df=scale['valid']
    summaries=[summarize('Derived comparison','All checkpoints',df)]
    summaries.extend(summarize('Derived comparison','Size: '+s,group) for s,group in df.groupby('size'))
    scale_overview=pd.DataFrame(summaries)
    scale_checkpoints=pd.DataFrame([{'model_label':k,'family':grp.family.iloc[0],
                                    'size':grp['size'].iloc[0],**paired_stats(grp)}
        for k,grp in df.groupby('model_label')]).sort_values(['family','size'])
    scale_family=pd.DataFrame([{'family':family,'size':size,**paired_stats(group)}
        for (family,size),group in df.groupby(['family','size'])]).sort_values(['family','size'])
    print('DERIVED: Main Large + Scale Small outcomes:')
    pretty(scale_overview,['group','pairs','baseline_n','cdea_n','baseline_pct','cdea_pct',
        'delta_pp','relative_gain_pct','cdea_only','baseline_only','mcnemar_p','task_ci_low','task_ci_high'])
    print('All eight checkpoints:')
    pretty(scale_checkpoints,['model_label','size','pairs','baseline_pct','cdea_pct',
        'delta_pp','cdea_only','baseline_only'])
    record_table('07_derived_size_overview',scale_overview)
    record_table('08_derived_eight_checkpoints',scale_checkpoints)
    record_table('09_derived_by_family_size',scale_family)
else:
    print('Both ZIP archives are required for combined size comparisons.')


In [ ]:
if scale is not None:
    s=scale_checkpoints.sort_values(['family','size']).copy()
    fig=grouped_bars(s,'model_label',[('baseline_pct','Baseline'),('cdea_pct','CDEA')],
                     'Derived model-size comparison: Main Large / Scale Small',rotation=30,ylim=(0,104))
    save_chart('03_derived_eight_checkpoints',fig)

    wide=scale_family.pivot(index='family',columns='size',values='delta_pp').sort_index()
    fig,ax=plt.subplots(figsize=(8,4.35))
    x=np.arange(len(wide));w=.36
    for i,(size,label) in enumerate([('small','3B-4B'),('large','8B-9B')]):
        bars=ax.bar(x+(i-.5)*w,wide[size],w,label=label)
        ax.bar_label(bars,fmt='%+.1f',fontsize=8,padding=2)
    ax.set_xticks(x,[f'{f}*' if f=='xlam2' else f for f in wide.index])
    ax.set_ylabel('CDEA - Baseline (percentage points)')
    ax.set_title('Derived model-size comparison: Effect by model family')
    ax.axhline(0,color='gray',lw=.9);ax.legend(frameon=False)
    ax.grid(axis='y',alpha=.15);ax.set_axisbelow(True)
    fig.text(.5,-.025,'* xLAM 3B and 8B use different backbones; not a controlled size effect.',
             ha='center',fontsize=8)
    fig.tight_layout()
    save_chart('04_derived_benefit_by_family_size',fig)


## 04. Model-size interaction and task-category heterogeneity

Within each model family, compare **CDEA minus Baseline** for Small (Scale) and Large (Main) on identical task IDs. Report the small-minus-large interaction, task-resampled sensitivity intervals, differences by task category, and an analysis excluding xLAM's backbone-confounded 3B/8B contrast. Results are exploratory because controller versions differ between source studies.


In [ ]:
size_interactions=pd.DataFrame()
size_sensitivity=pd.DataFrame()
scale_categories=pd.DataFrame()
shared_checkpoint_context=pd.DataFrame()
if scale is not None:
    df=scale['valid']
    scale_categories=pd.DataFrame([{'stratum':st,'size':sz,**paired_stats(grp)}
        for (st,sz),grp in df.groupby(['stratum','size'])]).sort_values(['stratum','size'])
    wide=df.pivot_table(index=['family','task_id'],columns='size',values='paired_delta',aggfunc='first')
    if {'small','large'}.issubset(wide.columns):
        common=wide.dropna(subset=['small','large']).reset_index()
        common['small_minus_large_pp']=100*(common['small']-common['large'])
        interactions=[]
        for family,grp in common.groupby('family'):
            tmp=grp.assign(paired_delta=grp.small_minus_large_pp/100)
            lo,hi=ci_task_bootstrap(tmp)
            interactions.append({'family':family,'matched_tasks':len(grp),
                  'small_minus_large_pp':grp.small_minus_large_pp.mean(),
                  'task_ci_low':lo,'task_ci_high':hi,
                  'different_backbone':family=='xlam2'})
        size_interactions=pd.DataFrame(interactions).sort_values('family')
        for name,groups in [('All families',common),
                            ('Excluding xLAM',common.loc[common.family!='xlam2'])]:
            if len(groups)==0:continue
            x=groups.assign(paired_delta=groups.small_minus_large_pp/100)
            lo,hi=ci_task_bootstrap(x)
            size_sensitivity=pd.concat([size_sensitivity,pd.DataFrame([{
                'comparison':name,'paired_family_task_units':len(groups),
                'unique_tasks':groups.task_id.nunique(),
                'small_minus_large_pp':groups.small_minus_large_pp.mean(),
                'task_ci_low':lo,'task_ci_high':hi}])],ignore_index=True)
    print('Within-family treatment-effect interactions (small CDEA effect - large CDEA effect):')
    pretty(size_interactions)
    print('Pooled across families only as a descriptive size-interaction sensitivity:')
    pretty(size_sensitivity)
    print('Derived treatment effects by task category and size:')
    pretty(scale_categories,['stratum','size','pairs','baseline_pct','cdea_pct','delta_pp',
                             'task_ci_low','task_ci_high'])
    record_table('10_derived_size_interactions',size_interactions)
    record_table('11_derived_sensitivity_no_xlam',size_sensitivity)
    record_table('12_derived_task_category_size',scale_categories)
    if not source_mapping.empty:
        shared_checkpoint_context=source_mapping.copy()
        print('Large versus Small data-source assignment:')
        pretty(shared_checkpoint_context)
        record_table('13_checkpoint_source_mapping',shared_checkpoint_context)

    order=['bank_information','cross_app_workflow','primitive_control']
    chart=scale_categories.pivot(index='stratum',columns='size',values='delta_pp').reindex(order)
    d=pd.DataFrame({'category':[s.replace('_',' ').title() for s in order],
                    'small':chart['small'].to_numpy(),'large':chart['large'].to_numpy()})
    fig=grouped_bars(d,'category',[('small','3B-4B'),('large','8B-9B')],
        'Main Large vs Scale Small: Effect by task category',
        ylabel='CDEA - Baseline (percentage points)')
    save_chart('05_derived_task_heterogeneity',fig)


## 05. CDEA mechanism and evidence acquisition

Analyze information-probe opportunities, comparable decisions, differing policies, stored evidence and trace events for the **two selected source cohorts** only: Main Large and Scale Small (128 pairs each). Raw Scale Large traces are excluded. Action divergence is not itself proof of correct information acquisition.


In [ ]:
mechanism_pairs=pd.DataFrame()
mechanism_by_divergence=pd.DataFrame()
trace_by_method=pd.DataFrame()
trace_examples=pd.DataFrame()
enriched_episodes={}
mechanism_rows=[]; diverge_rows=[]; trace_rows=[]; example_rows=[]

for study_name, study in SOURCES.items():
    if study is None: continue
    pairs=study['valid'].copy()
    raw=study['episodes'].copy()
    # Main has pair-level metrics duplicated across arms in results.csv; deduplicate by pair ID.
    if 'comparable_probe_decisions' not in pairs:
        for key in ('comparable_probe_decisions','policy_action_divergences'):
            if key in raw:
                once=raw.groupby('pair_id')[key].max().rename(key)
                pairs=pairs.merge(once,left_on='pair_id',right_index=True,how='left')
    for key in ('comparable_probe_decisions','policy_action_divergences'):
        if key not in pairs:pairs[key]=np.nan
        pairs[key]=pd.to_numeric(pairs[key],errors='coerce')
    measured=pairs['comparable_probe_decisions'].notna() & pairs['policy_action_divergences'].notna()
    mechanism_rows.append({'study':study_name,'valid_pairs':len(pairs),
        'pairs_with_diagnostics':int(measured.sum()),
        'pairs_with_comparable_decisions':int(pairs.comparable_probe_decisions.gt(0).sum()),
        'comparable_decisions_total':int(pairs.comparable_probe_decisions.fillna(0).sum()),
        'pairs_with_action_divergence':int(pairs.policy_action_divergences.gt(0).sum()),
        'divergent_decisions_total':int(pairs.policy_action_divergences.fillna(0).sum())})
    if measured.any():
        pairs['divergence_observed']=pairs.policy_action_divergences.gt(0)
        for has_div, group in pairs.loc[measured].groupby('divergence_observed'):
            diverge_rows.append({'study':study_name,'action_divergence':bool(has_div),**paired_stats(group)})

    # Use actual stored trace events (and preserve missing-trace counts).
    available=set()
    with zipfile.ZipFile(study['path']) as z:
        available=set(z.namelist())
        for _,ep_row in raw.iterrows():
            if ep_row.pair_id not in set(pairs.pair_id):continue
            rel=ep_row.get('episode_artifact')
            if not isinstance(rel,str) or not rel.endswith('/episode.json'):
                rel=f'episodes/{ep_row.pair_id}/{ep_row.method}/episode/episode.json'
            archive_entry=study['prefix']+rel.lstrip('/')
            item={'study':study_name,'pair_id':ep_row.pair_id,'method':ep_row.method,
                  'stratum':ep_row.stratum,'model_label':ep_row.model_label,
                  'official_success':bool(ep_row.official_success_raw),
                  'trace_available':False,'trace_steps':np.nan,'opportunity_steps':np.nan,
                  'selected_probe_steps':np.nan,'candidate_probes_seen':np.nan,
                  'override_steps':np.nan,'grounded_evidence_entries':np.nan,
                  'resolved_ledger_steps':np.nan,'repair_requests':np.nan,
                  'end_reason':ep_row.get('end_reason','unknown'),
                  'total_seconds_from_trace':np.nan}
            if archive_entry in available:
                ep=json.loads(z.read(archive_entry))
                trace=ep.get('trace')
                if isinstance(trace,list):
                    item['trace_available']=True
                    item['trace_steps']=len(trace)
                    item['opportunity_steps']=0
                    item['selected_probe_steps']=0
                    item['candidate_probes_seen']=0
                    item['override_steps']=0
                    item['grounded_evidence_entries']=0
                    item['resolved_ledger_steps']=0
                    for step in trace:
                        if not isinstance(step,dict):continue
                        item['opportunity_steps']+=int(bool(step.get('probe_opportunity')))
                        item['selected_probe_steps']+=int(bool(step.get('selected_probe')))
                        item['candidate_probes_seen']+=len(step.get('candidate_probes') or [])
                        item['override_steps']+=int(bool(step.get('policy_overrode_primary')))
                        observations=step.get('grounded_tentative_evidence') or []
                        item['grounded_evidence_entries']+=len(observations)
                        ledger=step.get('ledger') or {}
                        item['resolved_ledger_steps']+=int(any(
                            isinstance(v,dict) and str(v.get('state','')).upper() in ('SATISFIED','REFUTED')
                            for v in ledger.values()))
                        if step.get('selected_probe') and len(example_rows)<200:
                            selected=step['selected_probe'] or {}
                            example_rows.append({'study':study_name,'pair_id':ep_row.pair_id,
                                'method':ep_row.method,'task_id':f'{ep_row.task_group}/{ep_row.task}',
                                'step':step.get('step'),'selected_bid':selected.get('bid'),
                                'may_answer':','.join(selected.get('may_answer',[])),
                                'candidate_count':len(step.get('candidate_probes') or []),
                                'policy_override':bool(step.get('policy_overrode_primary')),
                                'tentative_evidence_entries':len(observations)})
                    item['repair_requests']=ep.get('repair_requests')
                    item['end_reason']=ep.get('end_reason') or item['end_reason']
                    item['total_seconds_from_trace']=ep.get('total_seconds')
            trace_rows.append(item)
    enriched_episodes[study_name]=raw.merge(
        pd.DataFrame([r for r in trace_rows if r['study']==study_name]),
        on=['pair_id','method','model_label','stratum'],how='left',suffixes=('','_trace'))

mechanism_pairs=pd.DataFrame(mechanism_rows)
mechanism_by_divergence=pd.DataFrame(diverge_rows)
trace_by_method=pd.DataFrame(trace_rows)
trace_examples=pd.DataFrame(example_rows)
print('Pair-level mechanism engagement:')
pretty(mechanism_pairs)
print('CDEA - Baseline success differences when policy actions did/did not diverge:')
pretty(mechanism_by_divergence,['study','action_divergence','pairs','cdea_only','baseline_only','delta_pp'])
trace_report=(trace_by_method.groupby(['study','method'],as_index=False).agg(
    episode_rows=('pair_id','size'), traces_present=('trace_available','sum'),
    trace_steps=('trace_steps','sum'),opportunity_steps=('opportunity_steps','sum'),
    selected_probe_steps=('selected_probe_steps','sum'),
    tentative_evidence_entries=('grounded_evidence_entries','sum'),
    resolved_ledger_steps=('resolved_ledger_steps','sum'))) if not trace_by_method.empty else pd.DataFrame()
print('Stored trace diagnostics (not independent truth verification):')
pretty(trace_report)
print('Representative selected-probe events (first five from each study):')
if not trace_examples.empty:
    pretty(trace_examples.groupby('study',group_keys=False).head(5))
for key,obj in [('14_mechanism_pairs',mechanism_pairs),('15_mechanism_by_divergence',mechanism_by_divergence),
                ('16_trace_indicators',trace_report),('17_probe_examples',trace_examples)]:
    record_table(key,obj)

# Two independent figures: one for Main, one for Scale. Each bar has the same unit (pairs).
for study_name in ('Main Large','Scale Small'):
    if study_name not in mechanism_pairs.study.values:continue
    row=mechanism_pairs.loc[mechanism_pairs.study==study_name].iloc[0]
    names=['Comparable probe decisions','Divergent policy actions']
    values=[row.pairs_with_comparable_decisions,row.pairs_with_action_divergence]
    fig,ax=plt.subplots(figsize=(7.7,4.0))
    bars=ax.barh(names,values,height=.48)
    ax.bar_label(bars, labels=[f'{int(v)} / {int(row.valid_pairs)} pairs' for v in values],padding=5)
    ax.set_xlim(0,max(int(row.valid_pairs)*1.10,max(values)*1.25,1))
    ax.set_xlabel('Matched pairs with at least one recorded event')
    ax.set_title(f'{study_name}: Mechanism engagement in matched pairs')
    ax.invert_yaxis();ax.grid(axis='x',alpha=.15);ax.set_axisbelow(True)
    fig.tight_layout()
    save_chart(f'0{6 if study_name=="Main Large" else 7}_{study_name.lower().replace(chr(32),chr(95))}_mechanism',fig)


## 06. Efficiency, failure patterns and trade-offs

Compare actions, model calls, inference time, duration, repairs and recorded failure reasons within selected matched pairs. Report all valid and both-successful pair subsets separately. Selected source cohorts are Main Large and Scale Small, excluding raw Scale Large entirely.


In [ ]:
paired_resources=pd.DataFrame()
resource_by_method=pd.DataFrame()
end_reasons=pd.DataFrame()
resource_rows=[]; method_rows=[]; failure_rows=[]; pair_steps={}
for study_name, study in SOURCES.items():
    if study is None:continue
    episode=enriched_episodes[study_name].copy()
    valid_ids=set(study['valid'].pair_id)
    episode=episode[episode.pair_id.isin(valid_ids)].copy()
    metric_alias={'steps':'steps','model_calls':'model_calls','model_seconds':'model_seconds',
                  'total_seconds':'total_seconds','repair_requests':'repair_requests'}
    if 'total_seconds' not in episode:
        episode['total_seconds']=episode.total_seconds_from_trace
    if 'repair_requests' not in episode:
        episode['repair_requests']=episode.repair_requests_trace if 'repair_requests_trace' in episode else np.nan
    if 'repair_requests' in episode and 'repair_requests_trace' in episode:
        episode['repair_requests']=pd.to_numeric(episode.repair_requests,errors='coerce').fillna(
            pd.to_numeric(episode.repair_requests_trace,errors='coerce'))
    for field in metric_alias:
        if field in episode:episode[field]=pd.to_numeric(episode[field],errors='coerce')
    for method,group in episode.groupby('method'):
        for metric in metric_alias:
            if metric not in group:continue
            arr=group[metric].dropna()
            if len(arr):
                method_rows.append({'study':study_name,'method':method,'metric':metric,
                                    'n':len(arr),'mean':arr.mean(),'median':arr.median()})
        failures=group.loc[group.official_success_raw.ne(True)]
        for reason,count in failures.end_reason.fillna('unknown').value_counts().items():
            failure_rows.append({'study':study_name,'method':method,
                                 'end_reason':reason,'unsuccessful_episodes':int(count)})
    for metric in metric_alias:
        if metric not in episode:continue
        pivot=episode.pivot(index='pair_id',columns='method',values=metric)
        if not {'Baseline','Proposed'}.issubset(pivot.columns):continue
        joined=(pivot.Proposed-pivot.Baseline).dropna()
        both=study['valid'].loc[study['valid'].baseline_success.eq(True) &
                                study['valid'].proposed_success.eq(True),'pair_id']
        for population,index in [('All valid',joined.index),
                                 ('Both succeeded',joined.index.intersection(both))]:
            diff=joined.loc[index]
            if len(diff):
                resource_rows.append({'study':study_name,'population':population,'metric':metric,
                                      'matched_pairs':len(diff),'median_cdea_minus_baseline':diff.median(),
                                      'mean_cdea_minus_baseline':diff.mean()})
        if metric=='steps':
            pair_steps[study_name]={'all':joined,'both':joined.loc[joined.index.intersection(both)]}

paired_resources=pd.DataFrame(resource_rows)
resource_by_method=pd.DataFrame(method_rows)
end_reasons=pd.DataFrame(failure_rows)
print('Within-pair CDEA - Baseline efficiency (negative values indicate smaller observed usage):')
pretty(paired_resources)
print('Episode-level resource summaries, kept separate by study and method:')
pretty(resource_by_method)
print('Unsuccessful episode end reasons:')
pretty(end_reasons)
record_table('18_paired_efficiency',paired_resources)
record_table('19_efficiency_by_method',resource_by_method)
record_table('20_failure_end_reasons',end_reasons)

for study_name in ('Main Large','Scale Small'):
    subset=paired_resources[(paired_resources.study==study_name)&(paired_resources.metric=='steps')]
    if subset.empty:continue
    subset=subset.set_index('population').reindex(['All valid','Both succeeded']).dropna(subset=['mean_cdea_minus_baseline'])
    if subset.empty:continue
    fig,ax=plt.subplots(figsize=(6.9,4.0))
    bars=ax.bar(subset.index,subset.mean_cdea_minus_baseline,width=.5)
    for bar, (_,row) in zip(bars,subset.iterrows()):
        y=bar.get_height()
        ax.annotate(f'{y:+.2f} steps (n={int(row.matched_pairs)})',
                    (bar.get_x()+bar.get_width()/2,y),xytext=(0,5 if y>=0 else 8),
                    textcoords='offset points',ha='center',fontsize=9,
                    color='white' if y<0 else 'black')
    ax.axhline(0,color='gray',lw=.9)
    ax.set_ylabel('Mean paired step difference (CDEA - Baseline)')
    ax.set_title(f'{study_name}: Mean browser-action trade-off')
    ax.grid(axis='y',alpha=.15);ax.set_axisbelow(True)
    fig.tight_layout()
    save_chart(f'0{8 if study_name=="Main Large" else 9}_{study_name.lower().replace(chr(32),chr(95))}_steps_efficiency',fig)


## 07. Results synthesis, source assignment and exports

The Main Study remains the sole source of Large outcomes, including the primary bank-information cohort. The derived size-comparison view combines its Large records with only the Scale Small records, without treating the Main outcomes as extra independent replications. Export separate PNG figures, analysis CSVs and source hashes/controller identities.


In [ ]:
import shutil

print('RESULTS SYNTHESIS - selected source cohorts')
if scale is not None:
    v=paired_stats(scale['valid'])
    print(f"  Derived size comparison: Baseline {v['baseline_n']}/{v['pairs']} ({v['baseline_pct']:.2f}%), "
          f"CDEA {v['cdea_n']}/{v['pairs']} ({v['cdea_pct']:.2f}%), delta {v['delta_pp']:+.2f} pp")
    print('  Main Large is used once as the Large group. The archived Scale Large records are excluded.')
for name, study in SOURCES.items():
    if study is None:continue
    st=paired_stats(study['valid'])
    print(f'  {name}: {st["pairs"]} matched valid pairs; Baseline {st["baseline_n"]}/{st["pairs"]} '
          f'({st["baseline_pct"]:.2f}%), CDEA {st["cdea_n"]}/{st["pairs"]} '
          f'({st["cdea_pct"]:.2f}%), delta {st["delta_pp"]:+.2f} pp')
if main is not None:
    p=paired_stats(main['valid'].loc[main['valid'].stratum.eq('bank_information')])
    print(f'  Main primary bank cohort: {p["delta_pp"]:+.1f} pp; exact McNemar p={p["mcnemar_p"]:.5f} '
          '(exploratory, unadjusted)')
if scale is not None and not size_sensitivity.empty:
    row=size_sensitivity[size_sensitivity.comparison=='Excluding xLAM']
    if len(row):
        r=row.iloc[0]
        print(f'  Size interaction excluding xLAM: {r.small_minus_large_pp:+.2f} pp '
              f'(task-bootstrap sensitivity interval {r.task_ci_low:+.2f} to {r.task_ci_high:+.2f} pp)')
if not trace_report.empty:
    print('  Trace observation: tentative evidence is a model/trace-level indicator, not verified correctness.')
print('  Main Large comes exclusively from Main; Small comes exclusively from Scale Small.')

for name, frame in TABLES.items():
    frame.to_csv(TAB/(name+'.csv'),index=False,float_format='%.8g')
metadata={'analysis_id':'cdea_main_large_plus_scale_small_v1',
          'studies':{name:{'file':study['path'].name,'sha256':study['sha256'],
                           'protocol':study['manifest']['protocol'],**study['counts']}
                     for name,study in SOURCES.items() if study is not None},
          'methods':['Baseline','Proposed (CDEA)'],
          'uncertainty':'Exact McNemar (unadjusted exploratory) and fixed-task cluster bootstrap sensitivity',
          'derived_size_comparison':{'large_source':'Main Study (8B-9B)',
                 'small_source':'Scale Study Small (3B-4B)',
                 'raw_scale_large_pairs_excluded':raw_excluded,
                 'valid_model_size_pairs':len(scale['valid']) if scale is not None else None},
          'controller_commits':{name:study['manifest'].get('controller_git_commit')
              for name,study in SOURCES.items() if study is not None},
          'notes':['The Large group reuses existing Main outcomes; never double-count Main',
                   'xLAM3B and xLAM8B differ in backbone',
                   'Action divergences are not direct evidence of correct information acquisition'],
          'figures':[p.name for p in FIGURES], 'tables':[k+'.csv' for k in TABLES]}
(ROOT/'sources_and_methods.json').write_text(json.dumps(metadata,indent=2),encoding='utf-8')
(ROOT/'README.txt').write_text(
    'CDEA research analysis output\n\nEvery PNG is one chart only.\n'
    'Tables and figures are derived from the source ZIPs named and fingerprinted in sources_and_methods.json.\n'
    'Size comparison uses Main Large and Scale Small; raw Scale Large is not used.\n'\n    'Source controller commits differ across Small and Large.\n',encoding='utf-8')
archive=shutil.make_archive(str(Path('cdea_research_analysis_outputs')),'zip',root_dir=ROOT)
print(f'Created {len(TABLES)} CSV tables, {len(FIGURES)} independent PNG figures')
print('Output directory:',ROOT.resolve())
print('Downloadable output archive:',Path(archive).resolve())
